[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/15_Transfer_Functions.ipynb)

# DiveLab

## Notebook 15 — Transfer Functions: Poles, Zeros and System Response

**Guiding question:** How can we describe the input-output behavior of a linear dynamical system with one algebraic function?

Notebook 14 introduced the Laplace transform.

Now we define:

\[
\boxed{G(s)=\frac{Y(s)}{U(s)}}
\]

under zero initial conditions.

This is the **transfer function**.

It will become our bridge from differential equations to classical control, PID and frequency response.

## Learning objectives

By the end of this notebook, you will be able to:

- derive a transfer function from a differential equation;
- understand why transfer functions assume zero initial conditions;
- identify poles and zeros;
- connect poles with eigenvalues and natural modes;
- predict stability from pole locations;
- interpret step and impulse responses;
- understand gain and time constants;
- derive a transfer function for a linearized diver model;
- see why feedback changes the poles of the system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

sp.init_printing()

# 1. From differential equation to transfer function

Consider:

\[
\dot y(t)+ay(t)=bu(t).
\]

With zero initial conditions:

\[
sY(s)+aY(s)=bU(s).
\]

Therefore:

\[
(s+a)Y(s)=bU(s).
\]

Divide by \(U(s)\):

\[
\boxed{
G(s)=\frac{Y(s)}{U(s)}=\frac{b}{s+a}
}
\]

The transfer function describes how the input is transformed into the output.

# 2. Why zero initial conditions?

Suppose:

\[
y(0)=y_0.
\]

Then:

\[
sY-y_0+aY=bU.
\]

So:

\[
Y(s)
=
\frac{b}{s+a}U(s)
+
\frac{y_0}{s+a}.
\]

There are now two contributions:

- response to the input;
- response to the initial state.

The transfer function isolates the **input-output relationship**:

\[
G(s)=\frac{Y(s)}{U(s)}
\]

by defining it with zero initial conditions.

# 3. Numerator and denominator

A general rational transfer function has the form:

\[
G(s)
=
\frac{b_ms^m+\cdots+b_1s+b_0}
     {a_ns^n+\cdots+a_1s+a_0}.
\]

The roots of the numerator are called **zeros**.

The roots of the denominator are called **poles**.

# 4. Poles

For:

\[
G(s)=\frac{1}{s+2},
\]

the denominator is zero when:

\[
s=-2.
\]

So the system has one pole at:

\[
\boxed{s=-2}.
\]

This corresponds to the natural mode:

\[
e^{-2t}.
\]

In [ ]:
s = sp.symbols("s")
G1 = 1/(s+2)

pole_G1 = sp.solve(sp.denom(G1), s)
pole_G1

# 5. Zeros

Consider:

\[
G(s)=\frac{s+1}{(s+2)(s+3)}.
\]

The zero is:

\[
s=-1.
\]

The poles are:

\[
s=-2,\qquad s=-3.
\]

Zeros do not represent autonomous modes in the same way poles do.

Instead, they shape how inputs propagate to outputs.

In [ ]:
G2 = (s+1)/((s+2)*(s+3))

zeros_G2 = sp.solve(sp.together(G2).as_numer_denom()[0], s)
poles_G2 = sp.solve(sp.together(G2).as_numer_denom()[1], s)

print("Zeros:", zeros_G2)
print("Poles:", poles_G2)

# 6. Pole-zero map

The \(s\)-plane gives us a visual summary.

Conventionally:

- poles are shown with \(\times\);
- zeros are shown with \(\circ\).

In [ ]:
poles = np.array([-2.0, -3.0])
zeros = np.array([-1.0])

plt.scatter(poles, np.zeros_like(poles), marker="x", s=100, label="Poles")
plt.scatter(zeros, np.zeros_like(zeros), marker="o", s=100, facecolors="none", label="Zeros")

plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")

plt.xlabel("Real part")
plt.ylabel("Imaginary part")
plt.title("Pole-zero map")
plt.grid(True)
plt.legend()
plt.show()

# 7. Poles and stability

For a continuous-time linear system:

### Stable

All poles lie in the left half-plane:

\[
\Re(s)<0.
\]

### Unstable

At least one pole lies in the right half-plane:

\[
\Re(s)>0.
\]

### Marginal cases

Poles on the imaginary axis require special care.

This is the transfer-function version of the eigenvalue stability criterion we already know.

# 8. Compare stable and unstable first-order systems

Consider:

\[
G_s(s)=\frac{1}{s+1}
\]

and:

\[
G_u(s)=\frac{1}{s-1}.
\]

Their poles are:

\[
-1,\qquad +1.
\]

Their natural modes are:

\[
e^{-t},\qquad e^{+t}.
\]

In [ ]:
t = np.linspace(0, 5, 500)

stable_mode = np.exp(-t)
unstable_mode = np.exp(t)

plt.plot(t, stable_mode, label="Pole at -1")
plt.plot(t, unstable_mode, label="Pole at +1")

plt.xlabel("Time [s]")
plt.ylabel("Mode amplitude")
plt.title("Pole location predicts natural behavior")
plt.grid(True)
plt.legend()
plt.show()

# 9. First-order transfer function in standard form

A common form is:

\[
G(s)
=
\frac{K}{\tau s+1}.
\]

Here:

- \(K\) is the static gain;
- \(\tau\) is the time constant.

The pole is:

\[
s=-\frac{1}{\tau}.
\]

So a smaller time constant moves the pole farther left and produces a faster response.

In [ ]:
taus = [0.5, 1.0, 2.0]
tt = np.linspace(0, 8, 800)

for tau in taus:
    y = 1 - np.exp(-tt/tau)
    plt.plot(tt, y, label=f"tau = {tau}")

plt.xlabel("Time [s]")
plt.ylabel("Step response")
plt.title("Time constant and response speed")
plt.grid(True)
plt.legend()
plt.show()

After one time constant:

\[
t=\tau,
\]

the response to a unit step has reached:

\[
1-e^{-1}\approx 0.632
\]

or about \(63.2\%\) of its final value.

In [ ]:
print(1 - np.exp(-1))

# 10. Static gain

For:

\[
G(s)=\frac{K}{\tau s+1},
\]

the steady response to a unit step is:

\[
y(\infty)=K.
\]

So \(K\) tells us how strongly a constant input changes the final output.

In [ ]:
K_values = [0.5, 1.0, 2.0]
tau = 1.0

for K in K_values:
    y = K*(1 - np.exp(-tt/tau))
    plt.plot(tt, y, label=f"K = {K}")

plt.xlabel("Time [s]")
plt.ylabel("Output")
plt.title("Static gain and final response")
plt.grid(True)
plt.legend()
plt.show()

# 11. Step response from the transfer function

For:

\[
G(s)=\frac{1}{s+2},
\]

a unit-step input has:

\[
U(s)=\frac{1}{s}.
\]

Therefore:

\[
Y(s)
=
G(s)U(s)
=
\frac{1}{s(s+2)}.
\]

Inverse Laplace gives:

\[
y(t)
=
\frac12(1-e^{-2t}).
\]

In [ ]:
Y_step = 1/(s*(s+2))
sp.apart(Y_step, s)

# 12. Impulse response

For an impulse input:

\[
U(s)=1.
\]

Therefore:

\[
Y(s)=G(s).
\]

For:

\[
G(s)=\frac{1}{s+2},
\]

the impulse response is:

\[
h(t)=e^{-2t}.
\]

The impulse response is therefore the inverse Laplace transform of the transfer function.

In [ ]:
t_sym = sp.symbols("t", positive=True)
h = sp.inverse_laplace_transform(G1, s, t_sym)
h

# 13. Second-order systems

A standard second-order transfer function is:

\[
G(s)
=
\frac{\omega_n^2}
{s^2+2\zeta\omega_n s+\omega_n^2}.
\]

Two parameters dominate the behavior:

- \(\omega_n\): natural frequency;
- \(\zeta\): damping ratio.

## Pole locations

The poles are:

\[
s
=
-\zeta\omega_n
\pm
\omega_n\sqrt{\zeta^2-1}.
\]

Different values of \(\zeta\) produce qualitatively different responses.

# 14. Underdamped, critically damped and overdamped

### Underdamped

\[
0<\zeta<1
\]

Complex poles and oscillatory response.

### Critically damped

\[
\zeta=1
\]

Fast non-oscillatory response.

### Overdamped

\[
\zeta>1
\]

Two real negative poles and slower non-oscillatory response.

In [ ]:
omega_n = 1.0

zetas = [0.2, 1.0, 2.0]

for zeta in zetas:
    roots = np.roots([1, 2*zeta*omega_n, omega_n**2])
    print(f"zeta={zeta}: poles={roots}")

# 15. Visualize second-order step responses

We can compute the step response numerically from:

\[
\ddot y
+
2\zeta\omega_n\dot y
+
\omega_n^2y
=
\omega_n^2u.
\]

In [ ]:
def simulate_second_order(zeta, omega_n=1.0, duration=15.0, dt=0.005):
    t = np.arange(0, duration+dt, dt)
    y = np.zeros_like(t)
    yd = np.zeros_like(t)

    for k in range(len(t)-1):
        u = 1.0

        ydd = (
            omega_n**2*u
            - 2*zeta*omega_n*yd[k]
            - omega_n**2*y[k]
        )

        yd[k+1] = yd[k] + ydd*dt
        y[k+1] = y[k] + yd[k+1]*dt

    return t, y

for zeta in [0.2, 1.0, 2.0]:
    t2, y2 = simulate_second_order(zeta)
    plt.plot(t2, y2, label=f"zeta = {zeta}")

plt.axhline(1, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Output")
plt.title("Second-order step responses")
plt.grid(True)
plt.legend()
plt.show()

The pole locations encode the shape of these responses.

This is why poles are one of the most important concepts in control engineering.

# 16. Return to the linearized diver

Use the simplified local model:

\[
\dot z=-v
\]

\[
\dot v=a_z z+b u.
\]

Eliminating \(v\):

\[
\ddot z+a_z z=-bu.
\]

With zero initial conditions:

\[
(s^2+a_z)Z(s)=-bU(s).
\]

Therefore:

\[
\boxed{
G(s)
=
\frac{Z(s)}{U(s)}
=
\frac{-b}{s^2+a_z}
}
\]

# 17. Compressible buoyancy produces an unstable pole

For the buoyancy instability:

\[
a_z<0.
\]

Write:

\[
a_z=-\alpha^2.
\]

Then:

\[
G(s)
=
\frac{-b}{s^2-\alpha^2}
=
\frac{-b}{(s-\alpha)(s+\alpha)}.
\]

The poles are:

\[
s=+\alpha,\qquad s=-\alpha.
\]

The positive pole produces an unstable mode.

In [ ]:
alpha = 0.12
b = 0.5

diver_poles = np.array([alpha, -alpha])

plt.scatter(diver_poles, np.zeros_like(diver_poles), marker="x", s=100)
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")

plt.xlabel("Real part of s")
plt.ylabel("Imaginary part of s")
plt.title("Open-loop diver poles")
plt.grid(True)
plt.show()

This is the same saddle point we previously found from:

- nonlinear physics;
- phase portraits;
- linearization;
- eigenvalues.

Now it appears as a pair of transfer-function poles.

Different mathematical languages are revealing the same physical instability.

# 18. Add proportional feedback

Suppose the diver uses a simplified proportional correction:

\[
u=K_p z.
\]

Substitute into:

\[
\ddot z+a_z z=-bu.
\]

Then:

\[
\ddot z+a_z z=-bK_pz.
\]

Therefore:

\[
\ddot z+(a_z+bK_p)z=0.
\]

Feedback changes the characteristic equation:

\[
s^2+a_z+bK_p=0.
\]

This is a fundamental control idea:

\[
\boxed{\text{feedback moves the closed-loop poles}}
\]

But proportional depth feedback alone does not necessarily create damping.

We will see why derivative action matters when we study PID control.

# 19. Add velocity feedback

Suppose:

\[
u=K_pz-K_dv.
\]

Since:

\[
v=-\dot z,
\]

we have:

\[
u=K_pz+K_d\dot z.
\]

Substitution gives:

\[
\ddot z+bK_d\dot z+(a_z+bK_p)z=0.
\]

Now the characteristic polynomial is:

\[
\boxed{
s^2+bK_ds+(a_z+bK_p)
}
\]

The \(K_d\) term introduces damping.

This gives a direct bridge to PID:

- proportional action changes restoring strength;
- derivative action changes damping;
- integral action will address persistent error.

Notebook 16 will develop this systematically.

# 20. Closed-loop transfer function

Consider the standard negative-feedback loop:

```text
        +          controller       plant
r ---->(Σ)----->     C(s)   ----->  G(s) -----> y
        - ^                                  |
          |__________________________________|
```

The error is:

\[
E=R-Y.
\]

And:

\[
Y=GC(R-Y).
\]

Therefore:

\[
Y+GCY=GCR.
\]

So:

\[
\boxed{
\frac{Y}{R}
=
\frac{GC}{1+GC}
}
\]

This is the closed-loop transfer function.

# 21. Characteristic equation

The closed-loop poles are determined by:

\[
\boxed{1+G(s)C(s)=0}.
\]

This equation is central to classical control.

Changing the controller \(C(s)\) changes the roots of this equation and therefore changes the closed-loop dynamics.

# 22. Example: first-order plant with proportional control

Let:

\[
G(s)=\frac{1}{s+1},
\qquad
C(s)=K_p.
\]

Then:

\[
T(s)
=
\frac{K_p}{s+1+K_p}.
\]

The closed-loop pole is:

\[
s=-(1+K_p).
\]

Increasing positive \(K_p\) moves the pole left and speeds up this simple system.

In [ ]:
Kps = [0.5, 2.0, 5.0]

for Kp in Kps:
    pole = -(1 + Kp)
    print(f"Kp={Kp:3.1f} -> closed-loop pole={pole:5.2f}")

# 23. But faster is not always better

Real systems contain:

- actuator limits;
- delay;
- sensor noise;
- neglected dynamics;
- nonlinearities.

Moving poles aggressively using high gain can therefore create new problems.

This reconnects with the delay, noise and saturation notebooks.

Control design is not simply "make the gain as large as possible."

# 24. Transfer functions and block diagrams

Transfer functions make interconnected systems easy to represent.

### Series

If:

\[
Y=G_2G_1U,
\]

then:

\[
G_{\text{series}}=G_1G_2.
\]

### Parallel

If outputs add:

\[
G_{\text{parallel}}=G_1+G_2.
\]

### Negative feedback

\[
G_{\text{closed}}
=
\frac{G}{1+GH}.
\]

This algebra is one reason transfer functions became foundational in control engineering.

# 25. State-space and transfer functions

For:

\[
\dot x=Ax+Bu
\]

\[
y=Cx+Du,
\]

the transfer function is:

\[
\boxed{
G(s)
=
C(sI-A)^{-1}B+D
}
\]

under zero initial conditions.

So state-space and transfer-function models are mathematically connected.

In [ ]:
A = sp.Matrix([[0, -1],
               [-0.04, 0]])

B = sp.Matrix([[0],
               [1]])

C = sp.Matrix([[1, 0]])
D = sp.Matrix([[0]])

I = sp.eye(2)

G_state = sp.simplify((C * (s*I - A).inv() * B + D)[0])
G_state

The denominator contains the characteristic dynamics of \(A\).

Its roots correspond to the eigenvalues of the state matrix, except in special cases where a mode may not appear in a particular input-output channel.

That observation will later connect transfer functions to controllability and observability.

# 26. Poles versus zeros: intuitive picture

A useful first intuition is:

### Poles

Describe where the system tends to have strong natural dynamical behavior.

They strongly determine:

- stability;
- decay;
- growth;
- oscillation.

### Zeros

Describe frequencies or modes where the input-output pathway is suppressed or reshaped.

Zeros can strongly affect transient behavior even when they do not represent autonomous modes.

# 27. Why transfer functions matter for DiveLab

We now have the language needed to ask classical-control questions:

- Where are the poles?
- Is the diver model stable?
- How does feedback move the poles?
- How quickly does the system respond?
- Does it oscillate?
- What is the steady-state error?
- How does it respond to periodic disturbances?

These questions lead directly to:

\[
\boxed{\text{PID control}}
\]

and then:

\[
\boxed{\text{frequency response}}.
\]

# Exercises

### 1. Derive a transfer function

For:

\[
3\dot y+6y=2u,
\]

derive:

\[
G(s)=\frac{Y(s)}{U(s)}.
\]

Identify:

- pole;
- static gain;
- time constant.

### 2. Pole locations

Find the poles of:

\[
G(s)=\frac{4}{s^2+5s+6}.
\]

Predict whether the system is stable before simulating it.

In [ ]:
# Your code here

### 3. Unstable system

Consider:

\[
G(s)=\frac{1}{s^2-s-2}.
\]

Factor the denominator and identify the unstable pole.

In [ ]:
# Your code here

### 4. Compare time constants

Plot unit-step responses for:

\[
\tau=0.25,\;1,\;4.
\]

Verify the \(63.2\%\) rule.

In [ ]:
# Your code here

### 5. Feedback pole movement

For:

\[
G(s)=\frac{1}{s+1}
\]

with proportional controller:

\[
C(s)=K_p,
\]

compute the closed-loop pole for several values of \(K_p\).

What happens to response speed?

# Challenge — derive the diver closed-loop poles

Start from:

\[
\ddot z+a_z z=-bu
\]

and use:

\[
u=K_pz+K_d\dot z.
\]

1. Derive the closed-loop characteristic polynomial.
2. Find its poles for several \(K_p,K_d\).
3. Plot them in the \(s\)-plane.
4. Simulate the corresponding time responses.
5. Relate the pole positions to the phase-space behavior seen in earlier notebooks.

In [ ]:
# Your code here

# Summary

In this notebook we introduced transfer functions:

\[
G(s)=\frac{Y(s)}{U(s)}.
\]

We learned that:

- transfer functions describe linear input-output dynamics under zero initial conditions;
- denominator roots are poles;
- numerator roots are zeros;
- poles are closely related to natural modes and eigenvalues;
- left-half-plane poles decay;
- right-half-plane poles grow;
- first-order poles determine time constants;
- second-order poles determine damping and oscillation;
- step and impulse responses reveal system behavior;
- feedback changes the closed-loop poles;
- state-space and transfer-function descriptions are connected.

For the diver model, the unstable saddle point appears again as a right-half-plane pole.

### Core insight

\[
\boxed{
\text{pole location}
\longrightarrow
\text{time-domain behavior}
}
\]

and:

\[
\boxed{
\text{feedback}
\longrightarrow
\text{pole movement}
\longrightarrow
\text{changed dynamics}
}
\]

### Next — Notebook 16

We are ready for **PID Control**.

We will build it progressively:

\[
P
\rightarrow
PI
\rightarrow
PD
\rightarrow
PID
\]

and interpret each component as a model of diver behavior:

- **P:** react to current depth error;
- **I:** remember persistent error;
- **D:** react to the trend / vertical velocity.

Then we will examine overshoot, damping, steady-state error, saturation and integral windup.